In [75]:
import pandas as pd

path ='/home/thomas/backup-sensor-data/dynamic_weight/Marzo/16 marzo - 22 marzo/1.xlsx'
df=pd.read_excel(path)


print(df.head())



len(df)


###only two isloated events were found in the whole month of march
# 2025-03-18 00:15:00
# 2025-03-18 00:33:00


       Id      StartTime      StartTimeStr  LaneNo                  LaneName  \
0  949997  1742080000000  15/03/2025 23:00       3  Corsia 2 - Marcia veloce   
1  949998  1742080000000  15/03/2025 23:00       2         Corsia 1 - Marcia   
2  949999  1742080000000  15/03/2025 23:00       3  Corsia 2 - Marcia veloce   
3  950000  1742080000000  15/03/2025 23:00       2         Corsia 1 - Marcia   
4  950001  1742080000000  15/03/2025 23:01       3  Corsia 2 - Marcia veloce   

   BaseClassId Scheme  ClassId  MoveStatus  FrontToFront  ...  WheelBase  \
0           21  EUR13        1           1        15.685  ...       2.68   
1            2  EUR13       12          -1        33.209  ...       4.31   
2           21  EUR13        1          -1        36.706  ...       2.64   
3           15  EUR13        9          -1        36.045  ...      12.19   
4           21  EUR13        1           1        52.295  ...       2.71   

   FrontOverhang  AxlesCount  MassUnit  VelocityUnit  Distance

100000

In [85]:
from datetime import datetime, timedelta
import time
from tqdm import tqdm

def find_isolated_timestamps(timestamps, gap_minutes=5):
    fmt = "%d/%m/%Y %H:%M"
    
    parsed = [
        ts if isinstance(ts, datetime) else datetime.strptime(ts.strip(), fmt)
        for ts in timestamps
    ]
    parsed= sorted(parsed)

    print(parsed[2])    
    isolated = []
    gap = timedelta(minutes=gap_minutes)
    total = len(parsed)

    def find_effective_neighbor(i, direction):
        """Walk in a direction, skipping identical timestamps, return first different one."""
        j = i + direction
        while 0 <= j < total:
            if parsed[j] != parsed[i]:
                return parsed[j]
            j += direction
        return None

    for i in tqdm(range(total), desc="Scanning timestamps"):
        #prev_current = parsed[i-1]
        current = parsed[i]
        
        left_neighbor  = find_effective_neighbor(i, -1)
        right_neighbor = find_effective_neighbor(i, +1)
        
        left_ok  = left_neighbor  is None or abs(current - left_neighbor)  > gap
        right_ok = right_neighbor is None or abs(current - right_neighbor) > gap
        
        if left_ok and right_ok:
            isolated.append(current)

    print(f"\nDone — {len(isolated)} isolated timestamp(s) found out of {total}.")
    return isolated



timestamps = df['StartTimeStr']



isolated = find_isolated_timestamps(timestamps, gap_minutes=5)

####3min
#2025-03-22 23:00:00

###4min
# timestamp
# 0 2025-03-17 23:00:00
# 1 2025-03-17 23:00:00
# 2 2025-03-18 00:15:00
# 3 2025-03-18 00:21:00
# 4 2025-03-18 00:26:00
# 5 2025-03-18 00:26:00
# 6 2025-03-18 00:33:00

2025-03-15 23:00:00


Scanning timestamps: 100%|██████████| 100000/100000 [00:00<00:00, 479948.00it/s]


Done — 2 isolated timestamp(s) found out of 100000.


In [95]:
isolated = pd.to_datetime(isolated, format="%m/%d/%Y %H:%M")


isolated.dt.strftime('%Y-%d-%m %H:%M:%S')
print(isolated)

AttributeError: 'DatetimeIndex' object has no attribute 'dt'

In [87]:

df_isolated_timestamps =pd.DataFrame({'timestamp': isolated})

print(df_isolated_timestamps)

print(df_isolated_timestamps.dtypes)


            timestamp
0 2025-03-18 00:15:00
1 2025-03-18 00:33:00
timestamp    datetime64[ns]
dtype: object


In [42]:
df_isolated_timestamps.to_csv('/home/thomas/industry_time_series/src/results/less_traffic/isolated_timestamps_5min.csv')


In [ ]:



df_isolated = pd.read_csv('/Users/thomas/Desktop/github_repos/industry_time_series/src/results/less_traffic/isolated_timestamps_5min.csv', index_col=0)

print(df_isolated.dtypes)
print(df_isolated.head())


timestamp    object
dtype: object
             timestamp
0  2025-01-03 00:58:00
1  2025-01-03 01:54:00
2  2025-03-03 00:16:00
3  2025-03-03 01:21:00
4  2025-03-18 00:15:00
             timestamp
0  2025-01-03 00:58:00
1  2025-01-03 01:54:00
2  2025-03-03 00:16:00
3  2025-03-03 01:21:00
4  2025-04-03 00:25:00
5  2025-04-03 01:48:00
6  2025-06-03 00:16:00
7  2025-06-03 01:31:00
8  2025-06-03 02:19:00
9  2025-07-03 00:08:00
timestamp    datetime64[ns]
dtype: object


In [161]:

df_isolated_timestamps = pd.merge(df_isolated, df_isolated_timestamps, on='timestamp', how='outer')

print(df_isolated_timestamps.tail())

df_isolated_timestamps.to_csv('/Users/thomas/Desktop/github_repos/industry_time_series/src/results/less_traffic/isolated_timestamps_5min.csv', index=True)



             timestamp
13 2025-03-07 01:50:00
14 2025-03-08 01:48:00
15 2025-03-08 01:56:00
16 2025-03-18 00:15:00
17 2025-03-18 00:33:00
